# Session 1: forces on a ramp, and friction as a limit
**A. Real-world problem.** A parked car on a hill, a box on a loading ramp, a book on a tilted
desk. Tilt the surface a little and nothing moves; tilt it more and suddenly it slides.
What decides the angle where the sliding starts?

**Plan.** (1) split the weight into two parts, (2) find the most friction the surface can give,
(3) compare, (4) draw the picture and the graph.

In [ ]:
import pathlib

import numpy as np

# where figures are saved (works as a script and in Jupyter/Colab)
OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

G = 9.81  # gravity, m/s^2

## Step 1: split the weight
The weight m*g points straight down. On a ramp tilted by an angle t, it is easier to use two
directions: along the slope and straight into the slope.

    into the slope (balanced by the normal force N):   N = m g cos t
    down the slope (what tries to make it slide):       F = m g sin t

Check at t = 0 (flat table): N = m g and F = 0. At t = 90 degrees (a wall): N = 0 and F = m g.

In [ ]:
def normal_force(m, theta_deg, g=G):
    """Push of the ramp on the block, perpendicular to the ramp (newtons)."""
    return m * g * np.cos(np.radians(theta_deg))


def along_slope(m, theta_deg, g=G):
    """Part of the weight pulling the block down the slope (newtons)."""
    return m * g * np.sin(np.radians(theta_deg))

## Step 2: friction is a limit
Friction does not have a fixed size. It pushes back exactly as much as needed to stop the block,
up to a largest value: mu * N, where mu ("mu", the friction coefficient) has no unit and
depends on the two surfaces. If the slope pull F is below mu*N, friction equals F and the net
force is zero. If F would exceed mu*N, friction can only supply mu*N and the rest accelerates
the block. (Session 1 uses one mu for both resting and sliding; Session 2 separates them.)

In [ ]:
def friction_limit(m, theta_deg, mu, g=G):
    """Largest friction the surface can give (newtons)."""
    return mu * normal_force(m, theta_deg, g)


def slides(theta_deg, mu):
    """True if the block begins to slide: m g sin t > mu m g cos t, i.e. tan t > mu."""
    return bool(np.tan(np.radians(theta_deg)) > mu)


def friction_force(m, theta_deg, mu, g=G):
    """Friction that actually acts: only as large as needed, never above the limit."""
    return min(along_slope(m, theta_deg, g), friction_limit(m, theta_deg, mu, g))


def net_force(m, theta_deg, mu, g=G):
    """Net force down the slope (newtons). Zero while the block holds."""
    return along_slope(m, theta_deg, g) - friction_force(m, theta_deg, mu, g)


def acceleration(m, theta_deg, mu, g=G):
    """Acceleration down the slope (m/s^2)."""
    return net_force(m, theta_deg, mu, g) / m


def critical_angle(mu):
    """Angle (degrees) where sliding begins: tan t = mu."""
    return float(np.degrees(np.arctan(mu)))

## Step 3: see it
Left: the force on a 5 kg block against angle. The slope pull climbs, the friction limit falls,
and they cross at the critical angle. Right: free-body diagrams just below and just above it.

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    m, mu = 5.0, 0.5
    for t in (20, 26, 27, 35):
        print(f"angle {t:5.2f} deg: N = {normal_force(m, t):6.2f} N, slope pull = {along_slope(m, t):6.2f} N, "
              f"limit = {friction_limit(m, t, mu):6.2f} N, net = {net_force(m, t, mu):5.2f} N, "
              f"slides = {slides(t, mu)}")
    print(f"critical angle for mu = {mu}: {critical_angle(mu):.2f} degrees")

    ang = np.linspace(0, 60, 241)
    fig, ax = plt.subplots(figsize=(6.4, 4.2))
    ax.plot(ang, along_slope(m, ang), label="slope pull m g sin t")
    ax.plot(ang, friction_limit(m, ang, mu), label="friction limit mu m g cos t")
    ax.plot(ang, normal_force(m, ang), ls=":", color="gray", label="normal force m g cos t")
    ax.axvline(critical_angle(mu), color="red", ls="--")
    ax.set_xlabel("ramp angle (degrees)")
    ax.set_ylabel("force (N)")
    ax.set_title("5 kg block, mu = 0.5: pull beats friction above 26.6 degrees")
    ax.grid(alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig(OUT / "forces_vs_angle.png", dpi=120)

    fig2, axs = plt.subplots(1, 2, figsize=(8, 3.8))
    for ax2, t in zip(axs, (20, 35)):
        a = np.radians(t)
        down, nrm = np.array([np.cos(a), -np.sin(a)]), np.array([np.sin(a), np.cos(a)])
        c = np.array([0.0, 0.0])
        ax2.plot([-down[0], down[0]], [-down[1], down[1]], "k", lw=2)   # the ramp surface
        s = 0.02
        for vec, name, col in ((np.array([0, -1.0]) * m * G, "weight", "tab:blue"),
                               (nrm * normal_force(m, t), "normal", "tab:green"),
                               (-down * friction_force(m, t, mu), "friction", "tab:red")):
            ax2.arrow(c[0], c[1], s * vec[0], s * vec[1], head_width=0.04, color=col, length_includes_head=True)
            ax2.text(c[0] + s * vec[0] * 1.1, c[1] + s * vec[1] * 1.1, name, color=col, fontsize=9)
        ax2.plot(*c, "ks", ms=12)
        ax2.set_xlim(-1, 1)
        ax2.set_ylim(-1.2, 1.0)
        ax2.set_aspect("equal")
        ax2.set_title(f"{t} degrees: net = {net_force(m, t, mu):.1f} N")
        ax2.set_xlabel("x (arrow scale 0.02 m per N)")
        ax2.set_ylabel("y")
    fig2.tight_layout()
    fig2.savefig(OUT / "free_body.png", dpi=120)
    print("saved figures in", OUT)

## Try it
Do the coding tasks, answer the four questions in the README (section F), then run the
checkpoint.

## Coding tasks
1. **Easy.** Change `mu` in `lesson.py` to 0.3 and 0.9 and predict the critical angle before you run it (use arctan).
2. **Stretch.** Write `forces_table(m, mu)` that prints N, the pull, the limit and the net force for angles 10, 20, 30, 40 and 50 degrees.
3. **Explain.** Explain, in two sentences and without a formula, why a heavy and a light block start to slide at the same angle.

## Assessment
- **A. Conceptual.** What determines whether a block at rest on a ramp begins to slide? Does the mass matter? Does the size of the block's bottom face matter?
- **B. Numerical.** An 8 kg block sits on a ramp at 15 degrees, mu = 0.35. Find N, the pull, the friction limit and the friction that acts. Does it slide? Sanity check the answer.
- **C. Graph and simulation interpretation.** On the force graph for a 5 kg block, the slope pull and the friction limit cross at 26.6 degrees for mu = 0.5. Where would they cross for mu = 0.8, and which curve moves? What does the friction curve do to the left of the crossing?
- **D. Design and optimisation challenge.** Design: a 20 kg box must stay put without a rope on a wooden ramp with mu = 0.45. Pick a maximum ramp angle with a safety margin, so that the friction needed is at most 80 percent of the limit. A 1.2 m rise is needed. How long must the ramp be? What is the trade-off?

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    checks.append(("flat table: N = m g and slope pull = 0",
                   abs(L.normal_force(5, 0) - 5 * 9.81) < 1e-9 and abs(L.along_slope(5, 0)) < 1e-9))
    checks.append(("vertical wall: N = 0 and slope pull = m g",
                   abs(L.normal_force(5, 90)) < 1e-9 and abs(L.along_slope(5, 90) - 5 * 9.81) < 1e-9))
    N, F, lim = L.normal_force(5, 20), L.along_slope(5, 20), L.friction_limit(5, 20, 0.5)
    checks.append(("worked example: 5 kg, 20 degrees, mu 0.5 gives N 46.09, pull 16.78, limit 23.05 N",
                   abs(N - 46.09) < 0.01 and abs(F - 16.78) < 0.01 and abs(lim - 23.05) < 0.01))
    checks.append(("worked example: the block holds and friction equals the pull (16.78 N)",
                   (not L.slides(20, 0.5)) and abs(L.friction_force(5, 20, 0.5) - F) < 1e-9
                   and abs(L.net_force(5, 20, 0.5)) < 1e-9))
    checks.append(("worked example: critical angle for mu 0.5 is 26.57 degrees", abs(L.critical_angle(0.5) - 26.565) < 0.01))
    checks.append(("hand calculation: 35 degrees, 5 kg, mu 0.5 gives net force 8.04 N and a = 1.61 m/s^2",
                   abs(L.net_force(5, 35, 0.5) - 8.04) < 0.01 and abs(L.acceleration(5, 35, 0.5) - 1.608) < 0.01))
    checks.append(("mass cancels: a is the same for 1 kg and 50 kg, but the forces scale with m",
                   abs(L.acceleration(1, 35, 0.5) - L.acceleration(50, 35, 0.5)) < 1e-9
                   and abs(L.net_force(50, 35, 0.5) / L.net_force(1, 35, 0.5) - 50) < 1e-9))
    checks.append(("sliding exactly starts above the critical angle for several mu",
                   all(L.slides(L.critical_angle(mu) + 0.1, mu) and not L.slides(L.critical_angle(mu) - 0.1, mu)
                       for mu in (0.1, 0.3, 0.7, 1.0))))
    checks.append(("edge case: mu = 0 slides at any angle above 0 and a = g sin t",
                   L.slides(1, 0.0) and abs(L.acceleration(3, 30, 0.0) - 9.81 * 0.5) < 1e-9))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
